Create a topic with 3 partitions:
kafka-topics.sh --bootstrap-server $BS --create --topic keys-demo --partitions 3

In [ ]:
"""
Same key, same patition: watch where each customer's events land."""

In [6]:
from confluent_kafka import Producer

In [13]:
producer = Producer({"bootstrap.servers": "localhost:9092"})
landed = {}

In [14]:
def on_delivery(err, msg):
    landed.setdefault(msg.key().decode(), set()).add(msg.partition())

In [ ]:
for i in range(40):
    customer = f"customer-{i % 4}"
    producer.produce("keys-demo", key=customer, value=f"event {i}", callback=on_delivery)

producer.flush()

for customer, parts in sorted(landed.items()):
    print(f"{customer} -> partition {sorted(parts)}")


customer-0 -> partition [1]
customer-1 -> partition [0]
customer-2 -> partition [0]
customer-3 -> partition [2]


> kafka-console-consumer.sh --bootstrap-server $BS --topic keys-demo --from-beginning --max-messages 12 --formatter-property print.key=true --formatter-property print.partition=true